# Step 4 – Ablation of the selection stages (Section 5.4)

Compares the two-stage order with single stages, the reverse order, mRMR, ReliefF and LASSO on GSE7390 (25 folds).

*Part of the code accompanying* **Explainable, Leakage-Free Gene Selection for Breast Cancer Histologic Grade** (Lamba, Munjal, Gigras). Run the notebooks from the folder that holds them, with the exported CSV files in `./data`.

In [ ]:
import time, pickle, numpy as np, pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from imblearn.over_sampling import SMOTEN
from skrebate import ReliefF
%run core.ipynb
%run pipeline.ipynb

REPEATS, FOLDS = 5, 5
CG = list(pd.read_csv("./common_genes.csv").iloc[:, 0])
X, y, P = load("GSE7390", genes=CG)
Xv = X.values
variants = ["CFS then CONS (proposed)", "CFS only", "CONS only", "CONS then CFS", "mRMR", "ReliefF", "LASSO"]
res = {v: {"f1": [], "mcc": [], "sets": []} for v in variants}

In [ ]:
def mi_vector(v, D):
    nbmax = int(max(D.max(), v.max())) + 1
    H = entropies(D, np.full(D.shape[1], nbmax))
    Hv = _H_from_counts(np.bincount(v).astype(float))
    su = su_vector_fast(v, D, nbmax, Hv, H)
    return su * (Hv + H) / 2.0  # MI = SU*(Hx+Hy)/2

In [ ]:
def mrmr(D, y, k):
    rel = mi_vector(y, D)
    sel = [int(np.argmax(rel))]
    red = np.zeros(D.shape[1])
    while len(sel) < k:
        red += mi_vector(D[:, sel[-1]], D)
        score = rel - red / len(sel)
        score[sel] = -np.inf
        sel.append(int(np.argmax(score)))
    return sel


import os, sys
BUDGET=float(os.environ.get("BUDGET",1e9)); os.makedirs("./abl", exist_ok=True)
t0 = time.time()
for rep in range(REPEATS):
    skf = StratifiedKFold(FOLDS, shuffle=True, random_state=rep)
    for k, (tr, te) in enumerate(skf.split(Xv, y)):
        fp=f'./abl/r{rep}_k{k}.pkl'
        if os.path.exists(fp): continue
        if time.time()-t0>BUDGET: print('BUDGET STOP'); sys.exit(0)
        res = {v: {'f1': [], 'mcc': [], 'sets': []} for v in variants}
        seed = rep * 10 + k
        cuts = mdl_cutpoints(Xv[tr], y[tr], 3)
        inf = np.array([j for j, c in enumerate(cuts) if len(c)])
        nb_all = np.array([len(cuts[j]) + 1 for j in inf])
        D0 = apply_cuts(Xv[tr][:, inf], [cuts[j] for j in inf])
        D, Dy = SMOTEN(sampling_strategy="not majority", k_neighbors=5, random_state=seed).fit_resample(D0, y[tr])
        Dte_all = apply_cuts(Xv[te][:, inf], [cuts[j] for j in inf])

        def evaluate(v, sel):
            sel = np.asarray(sel, int)
            m = HNB().fit(D[:, sel], Dy, nb_all[sel], 3)
            mm = metrics(y[te], m.predict(Dte_all[:, sel]))
            res[v]["f1"].append(mm["macro_f1"]); res[v]["mcc"].append(mm["mcc"])
            res[v]["sets"].append(set(X.columns[inf[sel]]))

        s_cfs, _ = cfs_bfs(D, Dy, nb_all)
        s_c2, _, _ = consistency_bfs(D[:, s_cfs], Dy, 3)
        prop = list(np.array(s_cfs)[s_c2]); kk = len(prop)
        evaluate(variants[0], prop)
        evaluate(variants[1], s_cfs)
        s_cons, _, _ = consistency_bfs(D, Dy, 3)
        evaluate(variants[2], s_cons)
        s_c3, _ = cfs_bfs(D[:, s_cons], Dy, nb_all[s_cons])
        evaluate(variants[3], list(np.array(s_cons)[s_c3]) if len(s_c3) else s_cons)
        evaluate(variants[4], mrmr(D, Dy, kk))
        rf = ReliefF(n_features_to_select=kk, n_neighbors=10, n_jobs=1).fit(D.astype(float), Dy)
        evaluate(variants[5], list(np.argsort(-rf.feature_importances_)[:kk]))
        # LASSO on continuous standardised training data (real samples), path search for ~kk genes
        Xi = Xv[tr][:, inf]
        Z = (Xi - Xi.mean(0)) / (Xi.std(0) + 1e-9)
        best = np.array([0])
        for C in np.logspace(-2.5, 0.5, 16):
            from sklearn.multiclass import OneVsRestClassifier
            lr = OneVsRestClassifier(LogisticRegression(penalty="l1", solver="liblinear", C=C, max_iter=2000, class_weight="balanced")).fit(Z, y[tr])
            w = np.abs(np.vstack([e.coef_ for e in lr.estimators_])).sum(0); nz = np.where(w > 0)[0]
            best = nz[np.argsort(-w[nz])][:kk] if len(nz) else best
            if len(nz) >= kk: break
        sel = list(best)
        evaluate(variants[6], sel)
        pickle.dump(res, open(fp,'wb'))
        print(rep, k, {v: round(res[v]["f1"][-1], 3) for v in variants}, "%.0fs" % (time.time() - t0), flush=True)

print("DONE")